# Avian Influenza Dataset Pipeline

Author: Alexander Maksiaev

Purpose: Create weekly dataset using NCBI Virus, Andersen Lab, and GISAID.

Notes: 
* This script only works in a Linux environment with bioconda and ncbi_datasets installed. 
* The directory where this script is housed should also house "utils.py". 
* All data from GISAID must be downloaded prior to running this script.

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
from itertools import islice
from datetime import datetime
from collections import defaultdict 
import zipfile

os.chdir("/data/maksiaevai/Avian_Flu/")
print(os.listdir())
import importlib
import pipeline_funcs
importlib.reload(pipeline_funcs)
from pipeline_funcs import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

['unassigned_trees_prep.ipynb', 'unassigned_check.ipynb', 'archive', 'other_mammals_counts_01_02_2026.ipynb', 'wendy_data.ipynb', 'genotype_host_table.ipynb', 'openapi3.docs.yaml', 'GISAID_Figures.ipynb', 'preprint_gisaid_check.ipynb', 'D1.1_sources.ipynb', 'human_case_by_genotype_lollipop_plot.R', 'pipeline.ipynb', 'usda_cat_list.ipynb', 'genoflu_check.ipynb', '.ipynb_checkpoints', 'B3_13_APR14_r2t.ipynb', 'utils.py', 'north_america_only.ipynb', 'paloma_keep.ipynb', 'bash', 'rejects.ipynb', 'update_metadata_v3.ipynb', 'feline_relabeling.ipynb', 'h5n1_cross_ref.ipynb', 'genoflu.yml', '.venv', 'deduplicator_fasta.ipynb', 'Step3_avian_flu_GISAID_v5.ipynb', 'genoflu.py', 'pipeline_funcs.py', 'concatenator_v2.ipynb', 'Step2_avian_flu_Andersen_v5.ipynb', 'Step1_avian_flu_NCBI_Virus_v5.ipynb', 'Paloma copy 2.ipynb', 'missing_metadata_barchart.ipynb', '__pycache__', 'tree_relabel_keep_bootstrap_aim.py', '11_01_2024_04_01_2025_non_D1_1_D1_3_IN_OH.ipynb', '.git', 'references', 'metadata_maker.i

In [2]:
# Directory paths and input

# Input
# browser = input("Browser (Firefox, Chrome, or Edge): ")
# sleep_time = input("Seconds to wait in between clicks (recommended 5): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date (format: MM-DD-YYYY): ")
# prev_end_date = input("End date of previous dataset (format: MM-DD-YYYY): ")
# serotype = input("Serotype (e.g. H5N1): ")
# serotypes = list(serotype)
# genotypes = input("Genotypes (separate with commas and no spaces in between genotypes): ")
# genotypes = genotypes.split(",")


# Dates and locations
browser = "Firefox"
sleep_time = "6"
locations = "Antarctica,North America,South America"
start_date = "11-01-2021"
end_date = "02-20-2026"
# prev_end_date = "01-09-2026"
date_range = start_date + "--" + end_date
# prev_date_range = start_date + "--" + prev_end_date

# Maintenance serotypes and genotypes
serotypes = ["H5N1"]
genotypes = ["B3.13", "D1.1", "D1.3"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"

# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
# references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"

home = "/data/maksiaevai/"
# downloads = "/home/maksiaevai/Downloads/"
references = "/data/maksiaevai/Avian_Flu/references/"

# downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 
# prev_downloads_saved = home + prev_date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 


complete_files = home + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
andersen = home + "avian-influenza/metadata/"
# ncbi_virus = complete_files + "NCBI_Virus/"
downloads = complete_files + "downloads/"

for folder in [complete_files, downloads]:
    if not os.path.exists(complete_files): # checking if the directory exists or not
        os.makedirs(complete_files) # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# All serotypes and genotypes
# serotype = ""
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])


## Downloading Data

### NCBI Virus

In [3]:
os.chdir(downloads)

zipfile_name = "alphainfluenza-after_11012021.zip"

In [4]:
# %%bash
# set -e
# module load ncbi-datasets # no need to run sinteractive if running from on demand # --released-after 01/01/2026 --include isolate-,isolate-name\ \ --fields accession,virus-infraspecific-strain,virus-infraspecific-isolate,geo-location,host-common-name,isolate-collection-date,segment
# # datasets summary virus genome accession PZ008639.1 --as-json-lines | dataformat tsv virus-genome 
# datasets download virus genome taxon "Alphainfluenzavirus influenzae" --released-after 11/01/2021 --filename alphainfluenza-after_11012021.zip
# exit

In [5]:
with zipfile.ZipFile(downloads + "alphainfluenza-after_11012021.zip", 'r') as zip_ref:
    zip_ref.extractall(downloads + "alphainfluenza-after_11012021/")

### Andersen

In [6]:
os.chdir(home + "avian-influenza/")

In [7]:
%%bash
git pull # remote set-url https://github.com/andersen-lab/avian-influenza.git

Already up to date.


In [8]:
os.chdir(andersen)
metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t", low_memory=False)
print(len(metadata)) 
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1]) # if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata
metadata = metadata[metadata["is_retracted"] == False]

print(len(metadata)) 

20686
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
20198


### GISAID

In [9]:
input("User downloaded GISAID data? (ENTER if yes)")

User downloaded GISAID data? (ENTER if yes) 


''

In [15]:
all_metadata_files = []
all_fasta_files = []

for dirpath, dirs, files in os.walk(downloads):
    for file in files:
        file_name = os.path.join(dirpath, file)
        print(file_name)

        # Now go through files and get contents
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name)
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states_ref) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)
    break 

/data/maksiaevai/11-01-2021--02-20-2026_Antarctica_North_America_South_America/downloads/all_alphainfluenza.zip
/data/maksiaevai/11-01-2021--02-20-2026_Antarctica_North_America_South_America/downloads/gisaid_epiflu_sequence.fasta


KeyboardInterrupt: 

In [ ]:
print(all_metadata_files[0])

In [12]:
# os.chdir(downloads)

# ncbi_virus_downloads = ncbi_virus + "Downloads/"
# if not os.path.exists(ncbi_virus_downloads): # checking if the directory exists or not
#     os.makedirs(ncbi_virus_downloads) # if the directory is not present then create it

# # Move downloaded files to saved downloads
# for dirpath, dirs, files in os.walk(ncbi_virus_downloads):
#     if len(files) != 0:
#         break 
#     else: # If we don't have any downloaded files
#         # Get files
#         open_ncbi_virus(browser, sleep_time, locations, start_date, end_date) # Will not work on BioWulf because of permissions. Sorry

#         # Re-try 
#         for dirpath, dirs, files in os.walk(downloads):
#             if len(files) > 0: # If we have any files that need to be moved
#                 for file in files:
#                     file_name = os.path.join(dirpath, file)
#                     destination_path = os.path.join(ncbi_virus_downloads, os.path.basename(file_name))
#                     try:
#                         shutil.move(file_name, destination_path)
#                     except:
#                         print("Error moving file", file_name)
#                         continue 
#             break 
#     break 